In [1]:
# ============================================================================
# PREREQUISITE SETUP - RUN THIS FIRST
# ============================================================================

# Install libraries
!pip install xgboost lightgbm catboost shap scikit-learn matplotlib seaborn plotly -q

# libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# ML Libraries
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.metrics import (classification_report, accuracy_score, precision_score,
                              recall_score, f1_score, confusion_matrix, roc_auc_score,
                              roc_curve, precision_recall_curve)
from sklearn.preprocessing import label_binarize
import joblib
import pickle
import json
from datetime import datetime
import time

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# ============================================================================
# DATA LOADING AND PREPARATION
# ============================================================================

# Load your dataset (update the path)
df = pd.read_csv('/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Pace 02 - model training /Data set/processed data set new.csv')  # UPDATE THIS PATH

# ============================================================================
# MODIFICATION START: Sample 1000 rows
# ============================================================================
TARGET_ROWS = 1000
if len(df) > TARGET_ROWS:

    try:
        df_sampled = df.groupby('job_title', group_keys=False).apply(
            lambda x: x.sample(int(np.rint(TARGET_ROWS * len(x) / len(df))), random_state=42)
        )
        # Ensure we don't exceed TARGET_ROWS due to rounding
        df_sampled = df_sampled.head(TARGET_ROWS)

        # If stratified sampling didn't get exactly 1000 rows or failed, fall back to simple sample
        if len(df_sampled) != TARGET_ROWS:
            df_sampled = df.sample(n=TARGET_ROWS, random_state=42).reset_index(drop=True)
            print(f" Fell back to simple random sampling for 1000 rows due to class imbalance.")

        df = df_sampled
    except Exception:
        df = df.sample(n=TARGET_ROWS, random_state=42).reset_index(drop=True)
        print(f"⚠️ Simple random sampling used for 1000 rows.")

print("\n" + "="*70)
print(f"DATASET OVERVIEW (REDUCED TO {len(df)} ROWS)")
print("="*70)
# ============================================================================
# MODIFICATION END
# ============================================================================

# Display basic info
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
print(df.head())

# Define features and target
feature_cols = [
    'search_country_encoded', 'job_level_encoded', 'job_type_onsite',
    'job_title_word_count_scaled', 'skills_word_count_scaled',
    'job_title_char_count_scaled', 'skills_char_count_scaled',
    'skills_list_count_scaled', 'title_tokens_count_scaled',
    'skills_tokens_count_scaled', 'job_level_encoded_scaled',
    'search_country_encoded_scaled',
    'has_skill_communication', 'has_skill_customer_service',
    'has_skill_teamwork', 'has_skill_leadership',
    'has_skill_communication_skills'
] + [f'tfidf_component_{i}' for i in range(20)]

# Prepare X and y
X = df[feature_cols].fillna(0)
y = df['job_title']

# 2. Handle rare classes
class_counts = y.value_counts()

y = y.apply(lambda x: x if class_counts[x] >= 2 else "Other")

# 3. Train-test split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n Features: {len(feature_cols)}")
print(f" Samples: {len(X)}") # Should be 1000
print(f"\n Target Distribution (after rare class handling):")
print(y.value_counts())


print(f"\n Training samples: {len(X_train)}") # Should be 800 (80% of 1000)
print(f" Test samples: {len(X_test)}")      # Should be 200 (20% of 1000)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 10.2 MB/s eta 0:00:00
Mounted at /content/drive
 Fell back to simple random sampling for 1000 rows due to class imbalance.

DATASET OVERVIEW (REDUCED TO 1000 ROWS)
Shape: (1000, 66)

Columns: ['job_link', 'last_processed_time', 'got_summary', 'got_ner', 'is_being_worked', 'job_title', 'company', 'job_location', 'first_seen', 'search_city', 'search_country', 'search_position', 'job_level', 'job_type', 'job_skills', 'job_title_clean', 'company_clean', 'job_skills_clean', 'job_location_clean', 'job_title_word_count', 'skills_word_count', 'job_title_char_count', 'skills_char_count', 'skills_list_count', 'has_skill_communication', 'has_skill_customer_service', 'has_skill_teamwork', 'has_skill_leadership', 'has_skill_communication_skills', 'job_title_tokens', 'job_skills_tokens', 'title_tokens_count', 'skills_tokens_count', 'combined_text_clean', 'job_level_encoded', 'job_type_onsite', 'search_country_encoded', 'job_title_word_count_sc

In [2]:
# ============================================================================
# MODEL 4: CATBOOST CLASSIFIER
# ============================================================================

print("TRAINING CATBOOST CLASSIFIER...")

from catboost import CatBoostClassifier

# Create model directory
model_dir = f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Pace 02 - model training /Notebooks/Model_04_CatBoost_Classifier.ipynb'

TRAINING CATBOOST CLASSIFIER...


In [3]:
# ============================================================================
# PART 1: EDA BEFORE TRAINING
# ============================================================================

print("\nPART 1: PRE-TRAINING EDA")

# 1. Categorical Features Analysis
categorical_features = ['search_country_encoded', 'job_level_encoded', 'job_type_onsite']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for idx, cat_feat in enumerate(categorical_features):
    value_counts = X_train[cat_feat].value_counts()
    axes[idx].bar(range(len(value_counts)), value_counts.values, color='orange', edgecolor='black')
    axes[idx].set_title(f'{cat_feat} Distribution', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Category', fontsize=10)
    axes[idx].set_ylabel('Count', fontsize=10)
    axes[idx].set_xticks(range(len(value_counts)))
    axes[idx].set_xticklabels(value_counts.index, rotation=45)
plt.suptitle('Categorical Features Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Results/Model 04 - outputs-categorical_features.png', dpi=300, bbox_inches='tight')
plt.close()

# 2. Numerical Features Summary
numerical_features = [col for col in feature_cols if col not in categorical_features]
num_stats = X_train[numerical_features].describe()
num_stats.to_csv(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Results/Model 04 - outputs-numerical_features_summary.csv')

plt.figure(figsize=(15, 10))
sns.heatmap(num_stats, annot=True, fmt='.2f', cmap='YlOrRd', cbar=True)
plt.title('Numerical Features Statistics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Results/Model 04 - outputs-02_numerical_stats.png', dpi=300, bbox_inches='tight')
plt.close()

# 3. Target vs Categorical Features
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for idx, cat_feat in enumerate(categorical_features):
    cross_tab = pd.crosstab(X_train[cat_feat], y_train, normalize='index') * 100
    cross_tab.plot(kind='bar', ax=axes[idx], stacked=False, color=['#ff9999', '#66b3ff', '#99ff99'])
    axes[idx].set_title(f'Target Distribution by {cat_feat}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel(cat_feat, fontsize=10)
    axes[idx].set_ylabel('Percentage (%)', fontsize=10)
    axes[idx].legend(title='Target', fontsize=8)
    axes[idx].tick_params(axis='x', rotation=45)
plt.suptitle('Target Distribution Across Categorical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Results/Model 04 - outputs-03_target_by_categories.png', dpi=300, bbox_inches='tight')
plt.close()

print("Pre-training EDA visualizations saved!")


PART 1: PRE-TRAINING EDA
Pre-training EDA visualizations saved!


In [5]:
# =============================================================================
# PART 2: HYPERPARAMETER TUNING (GPU SAFE & OPTIMIZED)
# =============================================================================

import time
import json
import joblib
import numpy as np
from datetime import datetime
from catboost import CatBoostClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import os

# =============================================================================
# SETUP
# =============================================================================

print("\nPART 2: HYPERPARAMETER TUNING (GPU Optimized, 3 Iterations)\n")

# ---- Encode target variable ----
label_encoder = LabelEncoder()
y_train_encoded_full = label_encoder.fit_transform(y_train)

# ---- Subsample 1000 samples for faster tuning ----
np.random.seed(42)
subset_size = min(1000, len(X_train))

if isinstance(X_train, pd.DataFrame):
    X_train_data = X_train.values
else:
    X_train_data = X_train

subset_idx = np.random.choice(len(X_train), size=subset_size, replace=False)
X_train_sub = X_train_data[subset_idx]
y_train_sub = y_train_encoded_full[subset_idx]

print(f"Subsampled dataset: {X_train_sub.shape[0]} samples used for tuning.")

# ---- Handle categorical features safely ----
categorical_features = []
try:
    cat_features_indices = [feature_cols.index(feat) for feat in categorical_features if feat in feature_cols]
except (NameError, ValueError):
    cat_features_indices = []
    print("Warning: No categorical features found. Using empty list.")

# =============================================================================
# DEFINE PARAMETER DISTRIBUTIONS
# =============================================================================

param_distributions = [
    # Iteration 1
    {
        'iterations': [100, 150, 200],
        'depth': [4, 6, 8],
        'learning_rate': [0.03, 0.07, 0.1],
        'l2_leaf_reg': [1, 3, 5]
    },
    # Iteration 2
    {
        'iterations': [150, 200, 250],
        'depth': [5, 6, 7],
        'learning_rate': [0.05, 0.1, 0.15],
        'l2_leaf_reg': [2, 3, 4],
        'border_count': [32, 64]
    },
    # Iteration 3
    {
        'iterations': [200, 250, 300],
        'depth': [6, 7],
        'learning_rate': [0.08, 0.1, 0.12],
        'l2_leaf_reg': [3, 4],
        'border_count': [64],
        'bagging_temperature': [0, 1]
    }
]

# =============================================================================
# RANDOMIZED SEARCH EXECUTION (GPU SAFE)
# =============================================================================

tuning_results = []
model_dir = "/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Results/Model_04"
os.makedirs(model_dir, exist_ok=True)

for iteration, param_dist in enumerate(param_distributions, 1):
    print(f"\n{'='*70}")
    print(f"Iteration {iteration}/3 - Randomized Search (GPU)")
    print(f"{'='*70}")

    base_model = CatBoostClassifier(
        random_state=42,
        verbose=False,
        cat_features=cat_features_indices,
        loss_function='MultiClass',
        task_type='GPU',
        bootstrap_type='Poisson'
    )

    random_search = RandomizedSearchCV(
        estimator=base_model,
        param_distributions=param_dist,
        n_iter=5,   # Reduced for memory-safe testing
        scoring='f1_weighted',
        cv=StratifiedKFold(n_splits=2, shuffle=True, random_state=42),  # 🔧 2 folds for speed
        n_jobs=1,   # GPU handles parallelism internally
        verbose=2,
        random_state=42,
        error_score='raise'
    )

    start_time = time.time()
    try:
        random_search.fit(X_train_sub, y_train_sub)
    except Exception as e:
        print(f"\nIteration {iteration} failed: {e}")
        continue
    end_time = time.time()

    iteration_result = {
        'iteration': iteration,
        'best_params': random_search.best_params_,
        'best_score': random_search.best_score_,
        'time_taken_sec': round(end_time - start_time, 2)
    }
    tuning_results.append(iteration_result)

    print(f"\nIteration {iteration} Results:")
    print(f"Best Parameters: {random_search.best_params_}")
    print(f"Best F1 Weighted Score: {random_search.best_score_:.4f}")
    print(f"⏱ Time Taken: {end_time - start_time:.2f} seconds")

# =============================================================================
# SAVE RESULTS
# =============================================================================

try:
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    output_path = os.path.join(model_dir, f"/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Results/Model 04 - outputs-Hyperparameter_Tuning_Optimized_{timestamp}.json")

    with open(output_path, 'w') as f:
        json.dump(tuning_results, f, indent=4)

    # Identify best overall iteration
    best_iteration = max(tuning_results, key=lambda x: x['best_score'])
    best_params = best_iteration['best_params']

    print(f"\nBest Overall Iteration: {best_iteration['iteration']}")
    print(f"Best Score: {best_iteration['best_score']:.4f}")
    print(f"Best Parameters:\n{json.dumps(best_params, indent=4)}")
    print(f"\nResults saved to: {output_path}")

except Exception as e:
    print(f"\nError saving tuning results: {e}")

# =============================================================================
# SAVE LABEL ENCODER
# =============================================================================

try:
    joblib.dump(label_encoder, os.path.join(model_dir, "LabelEncoder.pkl"))
    print("\nLabel encoder saved successfully.")
except Exception as e:
    print(f"\nCould not save label encoder: {e}")



PART 2: HYPERPARAMETER TUNING (GPU Optimized, 3 Iterations)

Subsampled dataset: 800 samples used for tuning.

Iteration 1/3 - Randomized Search (GPU)
Fitting 2 folds for each of 5 candidates, totalling 10 fits
[CV] END depth=6, iterations=100, l2_leaf_reg=3, learning_rate=0.03; total time=   2.2s
[CV] END depth=6, iterations=100, l2_leaf_reg=3, learning_rate=0.03; total time=   2.4s
[CV] END depth=4, iterations=100, l2_leaf_reg=1, learning_rate=0.03; total time=   2.8s
[CV] END depth=4, iterations=100, l2_leaf_reg=1, learning_rate=0.03; total time=   1.9s
[CV] END depth=4, iterations=200, l2_leaf_reg=3, learning_rate=0.07; total time=   2.0s
[CV] END depth=4, iterations=200, l2_leaf_reg=3, learning_rate=0.07; total time=   2.0s
[CV] END depth=6, iterations=100, l2_leaf_reg=3, learning_rate=0.07; total time=   2.2s
[CV] END depth=6, iterations=100, l2_leaf_reg=3, learning_rate=0.07; total time=   2.3s
[CV] END depth=4, iterations=200, l2_leaf_reg=1, learning_rate=0.03; total time=   4

In [6]:
# =============================================================================
# PART 3: CROSS-VALIDATION WITH BEST PARAMETERS
# =============================================================================

print("\nPART 3: CROSS-VALIDATION (3-Fold)")

# Initialize final CatBoost model with best hyperparameters
final_model = CatBoostClassifier(
    **best_params,
    random_state=42,
    verbose=False,
    cat_features=cat_features_indices
)

# Perform 5-fold cross-validation
cv_scores = cross_val_score(
    final_model,
    X_train,
    y_train,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='f1_weighted',
    n_jobs=1
)

# Print CV results
print(f"\nCV Scores: {cv_scores}")
print(f"Mean CV Score: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

# Plot CV scores
plt.figure(figsize=(10, 6))
plt.plot(
    range(1, 6),
    cv_scores,
    marker='*',
    linestyle='-',
    linewidth=2,
    markersize=15,
    color='darkorange'
)
plt.axhline(
    y=cv_scores.mean(),
    color='r',
    linestyle='--',
    label=f'Mean: {cv_scores.mean():.4f}'
)
plt.fill_between(
    range(1, 6),
    cv_scores.mean() - cv_scores.std(),
    cv_scores.mean() + cv_scores.std(),
    alpha=0.2,
    color='orange'
)
plt.xlabel('Fold', fontsize=12, fontweight='bold')
plt.ylabel('F1 Score', fontsize=12, fontweight='bold')
plt.title('CatBoost Cross-Validation Scores', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Save the plot
os.makedirs(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Results/Model 04 - outputs-visualizations', exist_ok=True)
plt.savefig(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Results/Model 04 - outputs-04_cv_scores.png', dpi=300, bbox_inches='tight')
plt.close()



PART 3: CROSS-VALIDATION (3-Fold)

CV Scores: [0.82844388 0.83433219 0.82842466 0.81360169 0.81734926]
Mean CV Score: 0.8244 (+/- 0.0077)


In [7]:
# =============================================================================
# PART 4: FINAL MODEL TRAINING
# =============================================================================

print("\nPART 4: FINAL MODEL TRAINING")

# Train the final model on the full training data
start_time = time.time()
final_model.fit(X_train, y_train)
train_time = time.time() - start_time
print(f"Model trained in {train_time:.2f}s")

# Predictions
y_train_pred = final_model.predict(X_train)
y_test_pred = final_model.predict(X_test)
y_test_proba = final_model.predict_proba(X_test)

# Training metrics
train_metrics = {
    'accuracy': accuracy_score(y_train, y_train_pred),
    'precision': precision_score(y_train, y_train_pred, average='weighted', zero_division=0),
    'recall': recall_score(y_train, y_train_pred, average='weighted', zero_division=0),
    'f1_score': f1_score(y_train, y_train_pred, average='weighted', zero_division=0)
}

# Testing metrics
test_metrics = {
    'accuracy': accuracy_score(y_test, y_test_pred),
    'precision': precision_score(y_test, y_test_pred, average='weighted', zero_division=0),
    'recall': recall_score(y_test, y_test_pred, average='weighted', zero_division=0),
    'f1_score': f1_score(y_test, y_test_pred, average='weighted', zero_division=0)
}

# Print metrics
print("\nTRAINING SET METRICS:")
for metric, value in train_metrics.items():
    print(f"  {metric.capitalize()}: {value:.4f}")

print("\nTEST SET METRICS:")
for metric, value in test_metrics.items():
    print(f"  {metric.capitalize()}: {value:.4f}")



PART 4: FINAL MODEL TRAINING
Model trained in 18.22s

TRAINING SET METRICS:
  Accuracy: 0.9962
  Precision: 0.9938
  Recall: 0.9962
  F1_score: 0.9948

TEST SET METRICS:
  Accuracy: 0.8800
  Precision: 0.7836
  Recall: 0.8800
  F1_score: 0.8278


In [8]:
# =============================================================================
# PART 5: POST-TRAINING VISUALIZATIONS
# =============================================================================

print("\nPART 5: POST-TRAINING VISUALIZATIONS")

# 1. Confusion Matrix
cm = confusion_matrix(y_test, y_test_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', cbar=True)
plt.title('CatBoost Confusion Matrix', fontsize=16, fontweight='bold')
plt.ylabel('True Label', fontsize=12, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Results/Model 04 - outputs-05_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.close()

# 2. Feature Importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False).head(20)

plt.figure(figsize=(12, 8))
plt.barh(feature_importance['feature'], feature_importance['importance'],
         color='darkorange', edgecolor='black')
plt.xlabel('Importance', fontsize=12, fontweight='bold')
plt.ylabel('Feature', fontsize=12, fontweight='bold')
plt.title('CatBoost Top 20 Feature Importance', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Results/Model 04 - outputs-06_feature_importance.png', dpi=300, bbox_inches='tight')
plt.close()

# Save feature importance to CSV
feature_importance.to_csv(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Results/Model 04 - outputs-feature_importance.csv', index=False)

# 3. Learning Curve (if available)
try:
    evals_result = final_model.get_evals_result()
    if evals_result:
        plt.figure(figsize=(12, 6))
        # Plot available metrics
        for key in evals_result.keys():
            for metric in evals_result[key].keys():
                plt.plot(evals_result[key][metric], label=f'{key} - {metric}')
        plt.xlabel('Iteration', fontsize=12, fontweight='bold')
        plt.ylabel('Metric Value', fontsize=12, fontweight='bold')
        plt.title('CatBoost Learning Curve', fontsize=14, fontweight='bold')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Results/Model 04 - outputs-07_learning_curve.png', dpi=300, bbox_inches='tight')
        plt.close()
except:
    print("Learning curve not available")

# 4. Metrics Comparison (Train vs Test)
metrics_comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
    'Train': [train_metrics['accuracy'], train_metrics['precision'],
              train_metrics['recall'], train_metrics['f1_score']],
    'Test': [test_metrics['accuracy'], test_metrics['precision'],
             test_metrics['recall'], test_metrics['f1_score']]
})

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(metrics_comparison['Metric']))
width = 0.35

bars1 = ax.bar(x - width/2, metrics_comparison['Train'], width,
               label='Train', color='bisque', edgecolor='black')
bars2 = ax.bar(x + width/2, metrics_comparison['Test'], width,
               label='Test', color='darkorange', edgecolor='black')

ax.set_xlabel('Metrics', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('CatBoost: Train vs Test Metrics Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_comparison['Metric'])
ax.legend()
ax.set_ylim([0, 1.1])

# Annotate bars with values
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom', fontsize=10)

plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(f'/content/drive/MyDrive/AIvy - AI Powered Job Recommendation System/Results/Model 04 - outputs-08_metrics_comparison.png', dpi=300, bbox_inches='tight')
plt.close()

print("All visualizations saved!")



PART 5: POST-TRAINING VISUALIZATIONS
All visualizations saved!
